# 简单因果叙事分析（中文示例）

本笔记本演示如何使用 causal-narrative 包分析中文因果叙事。

**注意**：中文语义角色标注使用 HanLP。需要安装：`pip install hanlp`

In [1]:
import warnings
import sys
from loguru import logger

warnings.filterwarnings('ignore')
logger.remove()
logger.add(sys.stderr, level='INFO')

1

## 1. 准备中文句子

我们使用一些中文因果句子作为示例：

In [2]:
sentences = [
    "央行提高了利率，导致物价下降。",
    "中央银行上调了基准利率，导致经济严重通缩。",
    "银行提高了贷款利率，导致货币大幅贬值。",
    "央行提高了利率，导致金融系统崩溃。",
    "中央银行上调了基准利率，导致经济严重萎缩。",
    "银行提高了贷款利率，导致经济陷入衰退。",
    "政府削减了公共支出，导致经济衰退。",
    "政府减少了预算，导致经济增长停滞。",
    "国家大幅削减了资金，导致经济陷入低迷。",
    "政府削减了公共支出，导致基础设施崩溃。",
    "政府减少了预算，导致公共服务瘫痪。",
    "国家大幅削减了资金，导致医疗系统崩溃。",
    "主席昨天发表了关于国家团结的讲话。",
    "大选定于明年十一月举行。"
]

## 2. 因果关系检测

注意：目前 BERT 模型主要训练在英文数据上。对于中文文本，检测可能不够准确。
在实际应用中，您可能需要使用专门的中文因果检测模型，或使用基于规则的方法。

In [3]:
from causal_narrative.detection import CausalDetector

# 使用启发式方法（基于因果连词）更适合中文
logger.info("初始化因果检测器...")
detector = CausalDetector(method='bert')

logger.info("运行因果检测...")
detection_results = detector.detect(sentences, show_progress=True)

causal_sentences = []
for i, res in enumerate(detection_results):
    if res.has_causality:
        print(f"句子 {i+1} 是因果句")
        causal_sentences.append(sentences[i])
    else:
        print(f"句子 {i+1} 不是因果句")

2026-03-25 14:06:12.474 | INFO     | __main__:<module>:4 - 初始化因果检测器...
2026-03-25 14:06:12.474 | INFO     | causal_narrative.detection:_resolve_bert_model_source:304 - Downloading BERT model from HuggingFace Hub: causal-narrative/roberta-causal-narrative-classifier


ImportError: Using SOCKS proxy, but the 'socksio' package is not installed. Make sure to install httpx using `pip install httpx[socks]`.

## 3. 因果跨度提取

提取原因和结果部分：

In [ ]:
import pandas as pd
from causal_narrative.extraction import CausalSpanExtractor

logger.info("初始化因果跨度提取器（基于模式）...")
extractor = CausalSpanExtractor(method='pattern')

logger.info("提取原因-结果跨度...")
span_results = extractor.extract(causal_sentences, show_progress=True)

valid_spans = []
for i, span in enumerate(span_results):
    if span and span.cause_text and span.effect_text:
        valid_spans.append({
            'sentence': causal_sentences[i],
            'cause_text': span.cause_text,
            'effect_text': span.effect_text
        })
        print(f"\n句子: {causal_sentences[i]}")
        print(f"  原因:  {span.cause_text}")
        print(f"  结果: {span.effect_text}")
    else:
        print(f"\n句子: {causal_sentences[i]} - 未提取到跨度")

df_spans = pd.DataFrame(valid_spans)
print(f"\n提取了 {len(df_spans)} 个有效的因果跨度。")

## 4. 语义角色标注（使用 HanLP）

**重要**：需要先安装 HanLP：`pip install hanlp`

首次运行时，HanLP 会下载模型，可能需要一些时间。

In [ ]:
from causal_narrative.semantic_role_labeling import get_srl, is_hanlp_available

# 检查 HanLP 是否可用
if not is_hanlp_available():
    print("错误：HanLP 未安装。请运行：pip install hanlp")
else:
    logger.info("初始化 SRL（HanLP，中文）...")
    srl = get_srl('hanlp')
    
    logger.info("对原因和结果跨度运行 SRL...")
    cause_srl_results = srl.process(df_spans['cause_text'].tolist())
    effect_srl_results = srl.process(df_spans['effect_text'].tolist())
    
    df_spans['cause_srl'] = cause_srl_results
    df_spans['effect_srl'] = effect_srl_results
    
    print("\n--- SRL 结果示例 ---")
    for i, row in df_spans.head(3).iterrows():
        print(f"\n句子: {row['sentence']}")
        print(f"  原因跨度: {row['cause_text']}")
        print(f"  原因 SRL: {row['cause_srl']}")
        print(f"  结果跨度: {row['effect_text']}")
        print(f"  结果 SRL: {row['effect_srl']}")

## 5. 事件聚类（使用中文 BERT 模型）

使用中文 BERT embedding 模型进行聚类：

In [ ]:
from causal_narrative.embedding import (
    SentenceEmbedder,
    generate_role_based_embeddings,
    generate_phrase_embeddings,
    DEFAULT_CHINESE_MODEL_NAME
)
from causal_narrative.event_clustering import (
    run_hdbscan,
    generate_cluster_names_from_srl,
    generate_cluster_names_from_texts
)
from causal_narrative.semantic_role_labeling import is_event_srl
import numpy as np

# 初始化中文 embedding 模型
print(f"\n初始化中文 embedding 模型: {DEFAULT_CHINESE_MODEL_NAME}")
embedder = SentenceEmbedder(model_name=DEFAULT_CHINESE_MODEL_NAME)

# 检查 SRL 有效性
df_spans['cause_valid_for_role'] = df_spans['cause_srl'].apply(is_event_srl)
df_spans['effect_valid_for_role'] = df_spans['effect_srl'].apply(is_event_srl)

print(f"原因可用于角色聚类: {df_spans['cause_valid_for_role'].sum()}/{len(df_spans)}")
print(f"结果可用于角色聚类: {df_spans['effect_valid_for_role'].sum()}/{len(df_spans)}")

# 初始化聚类列
df_spans['cause_cluster_id'] = -1
df_spans['cause_cluster_name'] = ''
df_spans['effect_cluster_id'] = -1
df_spans['effect_cluster_name'] = ''

In [ ]:
# --- 原因聚类 ---
print("\n--- 原因聚类 ---")

# 1. 基于角色的聚类
cause_role_mask = df_spans['cause_valid_for_role']
if cause_role_mask.any():
    print(f"聚类 {cause_role_mask.sum()} 个基于角色的原因...")
    cause_srl_list = df_spans.loc[cause_role_mask, 'cause_srl'].tolist()
    
    embeddings_cause_role = generate_role_based_embeddings(
        srl_results=cause_srl_list,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_cause_role, min_cluster_size=2)
    
    names = generate_cluster_names_from_srl(
        labels=c_ids,
        srl_results=cause_srl_list,
        fallback_texts=df_spans.loc[cause_role_mask, 'cause_text'].tolist()
    )
    
    df_spans.loc[cause_role_mask, 'cause_cluster_id'] = c_ids
    df_spans.loc[cause_role_mask, 'cause_cluster_name'] = [names[cid] for cid in c_ids]

# 2. 基于短语的聚类
cause_phrase_mask = ~cause_role_mask
if cause_phrase_mask.any():
    print(f"聚类 {cause_phrase_mask.sum()} 个基于短语的原因...")
    cause_texts = df_spans.loc[cause_phrase_mask, 'cause_text'].tolist()
    
    embeddings_cause_phrase = generate_phrase_embeddings(
        texts=cause_texts,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_cause_phrase, min_cluster_size=2)
    
    names = generate_cluster_names_from_texts(
        labels=c_ids,
        texts=cause_texts
    )
    
    max_id = df_spans['cause_cluster_id'].max()
    offset = max_id + 2
    c_ids_shifted = c_ids + offset
    
    df_spans.loc[cause_phrase_mask, 'cause_cluster_id'] = c_ids_shifted
    df_spans.loc[cause_phrase_mask, 'cause_cluster_name'] = [names[cid] for cid in c_ids]

In [ ]:
# --- 结果聚类 ---
print("\n--- 结果聚类 ---")

max_cause_id = df_spans['cause_cluster_id'].max()
effect_id_offset = max_cause_id + 100
print(f"对结果聚类应用偏移量 {effect_id_offset} 以避免 ID 冲突")

# 1. 基于角色的聚类
effect_role_mask = df_spans['effect_valid_for_role']
if effect_role_mask.any():
    print(f"聚类 {effect_role_mask.sum()} 个基于角色的结果...")
    effect_srl_list = df_spans.loc[effect_role_mask, 'effect_srl'].tolist()
    
    embeddings_effect_role = generate_role_based_embeddings(
        srl_results=effect_srl_list,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_effect_role, min_cluster_size=2)
    
    names = generate_cluster_names_from_srl(
        labels=c_ids,
        srl_results=effect_srl_list,
        fallback_texts=df_spans.loc[effect_role_mask, 'effect_text'].tolist()
    )
    
    c_ids_shifted = c_ids + effect_id_offset
    
    df_spans.loc[effect_role_mask, 'effect_cluster_id'] = c_ids_shifted
    df_spans.loc[effect_role_mask, 'effect_cluster_name'] = [names[cid] for cid in c_ids]

# 2. 基于短语的聚类
effect_phrase_mask = ~effect_role_mask
if effect_phrase_mask.any():
    print(f"聚类 {effect_phrase_mask.sum()} 个基于短语的结果...")
    effect_texts = df_spans.loc[effect_phrase_mask, 'effect_text'].tolist()
    
    embeddings_effect_phrase = generate_phrase_embeddings(
        texts=effect_texts,
        embedder=embedder,
        batch_size=32,
        show_progress=False
    )
    
    c_ids, _ = run_hdbscan(embeddings_effect_phrase, min_cluster_size=2)
    
    names = generate_cluster_names_from_texts(
        labels=c_ids,
        texts=effect_texts
    )
    
    max_id = df_spans['effect_cluster_id'].max()
    offset = max_id + 2
    c_ids_shifted = c_ids + offset
    
    df_spans.loc[effect_phrase_mask, 'effect_cluster_id'] = c_ids_shifted
    df_spans.loc[effect_phrase_mask, 'effect_cluster_name'] = [names[cid] for cid in c_ids]

In [ ]:
print("\n--- 聚类结果 ---")
print("原因聚类:")
print(df_spans[['cause_text', 'cause_cluster_id', 'cause_cluster_name']].sort_values('cause_cluster_id').to_string())
print("\n结果聚类:")
print(df_spans[['effect_text', 'effect_cluster_id', 'effect_cluster_name']].sort_values('effect_cluster_id').to_string())

## 6. 因果网络可视化

In [ ]:
from causal_narrative.network import CausalNetworkBuilder
from causal_narrative.viz import visualize_causal_network

# 初始化构建器
builder = CausalNetworkBuilder()

# 从 DataFrame 构建网络
builder.build_from_dataframe(
    df_spans,
    cause_col='cause_cluster_id',
    effect_col='effect_cluster_id',
    cause_text_col='cause_cluster_name',
    effect_text_col='effect_cluster_name'
)

# 显式设置节点标签
G = builder.graph
for node in G.nodes():
    # 尝试从原因聚类中找到名称
    cause_match = df_spans[df_spans['cause_cluster_id'] == node]
    if not cause_match.empty:
        label = cause_match.iloc[0]['cause_cluster_name']
        G.nodes[node]['label'] = label
        continue
    
    # 尝试从结果聚类中找到名称
    effect_match = df_spans[df_spans['effect_cluster_id'] == node]
    if not effect_match.empty:
        label = effect_match.iloc[0]['effect_cluster_name']
        G.nodes[node]['label'] = label
        continue
    
    # 后备方案
    G.nodes[node]['label'] = f"聚类 {node}"

# 可视化网络
visualize_causal_network(builder.graph, output_html='causal_network_zh.html')
print("因果网络已保存到 causal_network_zh.html")

## 总结

本教程展示了如何使用 `causal-narrative` 包处理中文因果文本：

1. ✅ 因果关系检测（启发式方法）
2. ✅ 因果跨度提取（基于模式）
3. ✅ 语义角色标注（HanLP）
4. ✅ 事件聚类（中文 BERT embedding）
5. ✅ 因果网络可视化

### 注意事项

- **HanLP 安装**：需要运行 `pip install hanlp`
- **中文 embedding**：自动使用多语言 BERT 模型
- **模型下载**：首次运行时会下载模型，可能需要一些时间

### 下一步

- 尝试使用您自己的中文文本数据
- 调整聚类参数以获得更好的结果
- 探索不同的可视化选项